# 实验四：AMP 混合精度与 Warmup 调优

混合精度可以降低显存占用并提升计算吞吐，但早期训练更容易出现 loss 波动。Warmup 的作用是在训练初期逐步放大学习率，让模型、优化器和梯度尺度稳定进入主训练阶段。

## AMP 的三个关键点

<table style="margin-left: 0; margin-right: auto; text-align: left;">
  <thead>
    <tr>
      <th style="text-align: left;">关键点</th>
      <th style="text-align: left;">说明</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <td style="text-align: left;"><code>autocast</code></td>
      <td style="text-align: left;">自动选择 FP16/BF16/FP32 执行合适算子</td>
    </tr>
    <tr>
      <td style="text-align: left;"><code>GradScaler</code></td>
      <td style="text-align: left;">对 loss 进行缩放，降低梯度下溢风险</td>
    </tr>
    <tr>
      <td style="text-align: left;">fallback</td>
      <td style="text-align: left;">不适合低精度的算子仍保留 FP32</td>
    </tr>
  </tbody>
</table>

在 Ascend NPU 上，训练脚本优先使用 `torch_npu.npu.amp`。如果环境不可用，会退化到 CPU 或 CUDA 的对应逻辑。

In [ ]:
# ====== 1. AMP 训练步模板 ======
from contextlib import nullcontext

def amp_step_template(model, images, targets, optimizer, scaler, autocast, loss_fn):
    optimizer.zero_grad(set_to_none=True)
    with autocast():
        pred = model(images)
        loss = loss_fn(pred, targets)
    if scaler is not None:
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
    else:
        loss.backward()
        optimizer.step()
    return loss

print('完整实现见 src/scripts/train_yolo_ddp_amp.py 中的 make_amp 和训练循环；单卡入口为 src/scripts/train_yolo_single_npu_amp.py。')

## Warmup + Cosine 学习率

本实验采用两段式学习率：

1. Warmup 阶段：从接近 0 的学习率线性升到基础学习率。
2. Cosine 阶段：从基础学习率平滑下降到最小学习率。

当前是单卡 910B3 训练，学习率先围绕 `batch_size=16` 建立稳定基线；当 batch size 增大到 24 或 32 时，可以小幅提高学习率，但每次只改一个变量，并配合 Warmup 观察 loss 是否稳定。

In [ ]:
# ====== 2. 可视化 Warmup + Cosine 学习率曲线 ======
import math
import matplotlib.pyplot as plt

base_lr = 0.01
min_lr = 0.0001
epochs = 100
steps_per_epoch = 100
warmup_epochs = 3

def lr_at(global_step):
    warmup_steps = warmup_epochs * steps_per_epoch
    total_steps = epochs * steps_per_epoch
    if global_step <= warmup_steps:
        return base_lr * global_step / warmup_steps
    progress = (global_step - warmup_steps) / (total_steps - warmup_steps)
    return min_lr + 0.5 * (base_lr - min_lr) * (1 + math.cos(math.pi * progress))

xs = list(range(1, epochs * steps_per_epoch + 1))
ys = [lr_at(x) for x in xs]
plt.figure(figsize=(10, 4))
plt.plot(xs, ys, linewidth=2)
plt.axvline(warmup_epochs * steps_per_epoch, color='tab:red', linestyle='--', label='warmup end')
plt.xlabel('global step')
plt.ylabel('learning rate')
plt.title('Warmup + Cosine Learning Rate')
plt.grid(alpha=0.3)
plt.legend()
plt.show()

In [ ]:
# ====== 3. 单卡 batch size 与学习率试探建议 ======
base_batch = 16
base_lr = 0.01

def scaled_lr(batch_size):
    # 单卡实验中不要激进线性放大学习率，先使用 sqrt 缩放作为保守参考。
    lr = base_lr * (batch_size / base_batch) ** 0.5
    return lr

for batch_size in [16, 24, 32]:
    lr = scaled_lr(batch_size)
    print(f'batch_size={batch_size:2d}  suggested_lr={lr:.4f}')

print('\n如果 loss 在 warmup 后仍明显发散，优先降低学习率或增加 warmup_epochs。')

## 超参数调优顺序

<table style="margin-left: 0; margin-right: auto; text-align: left;">
  <thead>
    <tr>
      <th style="text-align: left;">现象</th>
      <th style="text-align: left;">优先调整</th>
      <th style="text-align: left;">建议</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <td style="text-align: left;">前几百 step loss 发散</td>
      <td style="text-align: left;">Warmup / LR</td>
      <td style="text-align: left;">增加 <code>warmup_epochs</code>，学习率减半</td>
    </tr>
    <tr>
      <td style="text-align: left;">显存不足</td>
      <td style="text-align: left;">Batch / AMP / 累积</td>
      <td style="text-align: left;">开启 AMP，降低 batch size，必要时增加梯度累积</td>
    </tr>
    <tr>
      <td style="text-align: left;">NPU 利用率波动</td>
      <td style="text-align: left;">DataLoader / Host</td>
      <td style="text-align: left;">扫描 <code>workers=4/8/12</code>，检查 data gap 和 Host API 调度</td>
    </tr>
    <tr>
      <td style="text-align: left;">精度下降明显</td>
      <td style="text-align: left;">AMP 白名单 / LR</td>
      <td style="text-align: left;">对敏感 loss 保持 FP32，降低学习率</td>
    </tr>
    <tr>
      <td style="text-align: left;">验证集波动大</td>
      <td style="text-align: left;">Batch / seed</td>
      <td style="text-align: left;">固定 seed，延长训练或增加评估间隔</td>
    </tr>
  </tbody>
</table>

---
## 本章小结

AMP 解决吞吐和显存问题，Warmup 解决 batch size 变大后的早期稳定性问题。下一章将把这些策略放入单卡 NPU 训练命令中完整运行。

## 课后练习

请根据本节实验内容完成以下练习。题型包含单选题、多选题、判断题、填空题、简答题和代码设计题。

1. (单选题) AMP 混合精度训练的主要目的是什么？
   - A. 减少训练代码行数
   - B. 在保持精度可接受的前提下降低显存占用并提升计算效率
   - C. 把图片转为 XML
   - D. 替代优化器

2. (单选题) Warmup 学习率策略的核心作用是？
   - A. 训练初期逐步增大学习率，降低不稳定风险
   - B. 永久冻结模型参数
   - C. 删除低分目标框
   - D. 关闭 NPU

3. (单选题) 当训练出现 NPU 显存不足时，通常最先尝试调整的是？
   - A. 增大 batch size
   - B. 减小 batch size
   - C. 删除验证集
   - D. 关闭日志目录

4. (单选题) DataLoader workers 主要影响哪一部分？
   - A. 数据读取与预处理并行度
   - B. 模型类别名称
   - C. Git 远端地址
   - D. ATC 转换精度

5. (多选题) AMP 训练通常会涉及哪些组件或概念？
   - A. autocast
   - B. GradScaler
   - C. float16/bfloat16
   - D. 反向传播缩放

6. (多选题) 进行超参数调优时，应优先记录哪些指标？
   - A. loss 曲线
   - B. throughput 或 step time
   - C. 显存/NPU 使用情况
   - D. checkpoint 是否正常保存

7. (多选题) 训练不稳定时，可以尝试哪些调整？
   - A. 降低学习率
   - B. 增加 warmup epoch
   - C. 关闭或调整 AMP
   - D. 检查标注数据是否异常

8. (判断题) AMP 一定会让所有模型训练结果完全不变。

9. (判断题) 调优时一次只改一个关键参数，更容易判断变化来源。

10. (填空题) 训练初期逐步提升学习率的策略通常称为 `____`。

11. (填空题) 增大 `batch_size` 往往会提高吞吐，但也会增加 `____` 占用。

12. (简答题) 为什么不能只用 throughput 判断训练是否更好？

13. (简答题) Warmup 与 Cosine 学习率组合有什么意义？

14. (简答题) 如果开启 AMP 后 loss 变成 NaN，应如何排查？

15. (代码设计题) 写出 AMP 训练中 autocast 和 GradScaler 的核心用法。

> 参考答案见 answer/03.05_amp_warmup_tuning_answer.ipynb。
